# One crop recipe for every clean row: Tables 2, 8 and 14, plus feature-extraction timing

Reviewer 1, Concern 4: the clean 70-label probe AUC appeared as 0.986 (Tables 2, 8), 0.985 (Tables 9, 11) and
0.983 (Table 14). Tables 2 and 14 came from notebook 11 (its own crops, no JPEG round trip, `timm` DINO); Table 8
from crops rebuilt with a slightly different recipe. This notebook:

1. finds the crop recipe that **reproduces the cached features** behind Tables 9–11 (stops if none does);
2. rebuilds **Table 2** (DINOv1 from the cached features; DINOv2 and CLIP on the same cached crops),
   **Table 14** (mask perturbations) and **Table 8** (synthetic clutter) with that one recipe;
3. **times** crop building and DINO feature extraction for all 13,084 images (Reviewer 2, Comment 5);
4. lists every file in the dataset folder, to explain the 13,084 vs 13,085 count on Mendeley.

Output: `results_revision_v1/regen_tables_2_8_14.json` — upload it. Runtime ~35 min on a Colab GPU.

In [ ]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
EJSON      = f'{RES}/E_distillation_5rep.json'
OUT        = f'{RES}/regen_tables_2_8_14.json'
DEPLOY     = f'{RES}/deploy'
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache_r'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
ERODE_K, DILATE_K, SHIFT_FRAC = 11, 11, 0.12          # Table 14, as in notebook 11
ALPHAS = (0.0, 0.2, 0.4, 0.6)                          # Table 8


In [ ]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
!pip -q install timm
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(DEPLOY, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


In [ ]:
# CELL 1b — every file in the dataset folder (explains 13,084 here vs 13,085 on Mendeley)
from collections import Counter
allf = [os.path.join(r, f) for r, _, fs in os.walk(TEA_DRIVE) for f in fs]
ext = Counter(os.path.splitext(f)[1].lower() for f in allf)
print('all files: %d | by extension: %s' % (len(allf), dict(ext)))
others = [f for f in allf if not f.lower().endswith(IMG_EXT)]
print('files NOT read as images (%d):' % len(others)); [print('   ', os.path.relpath(f, TEA_DRIVE)) for f in others[:20]]
bad = []
for f in [f for f in allf if f.lower().endswith(IMG_EXT)]:
    try: Image.open(f).verify()
    except Exception as e: bad.append((os.path.relpath(f, TEA_DRIVE), type(e).__name__))
print('image files that fail to open: %d %s' % (len(bad), bad[:5]))
FILES = dict(n_all=len(allf), by_ext=dict(ext), not_images=[os.path.relpath(f, TEA_DRIVE) for f in others], unreadable=bad)


In [ ]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


In [ ]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
Xtr_c, Xte_c = F['Xtr'], F['Xte']
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


In [ ]:
# CELL 4 — two candidate crop recipes (both end with the JPEG q95 round trip of the cache)
# A: notebook 11 (flood-fill hole filling, integer padding, PIL default resize) — the code that built the cache
# B: reproduction notebook fallback (ndimage hole filling, float padding, bilinear resize)
def mask_A(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV); v, s = hsv[:, :, 2], hsv[:, :, 1]
    _, mv = cv2.threshold(v, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = ((mv > 0) | (ms > 0)).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1: return np.ones(rgb.shape[:2], np.uint8)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA])); m = (lab == k).astype(np.uint8)
    ff = m.copy(); h, w = m.shape
    cv2.floodFill(ff, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 1)
    return (m | (1 - ff)).astype(np.uint8)
def crop_A(rgb, m, pad=CROP_PAD):
    ys, xs = np.where(m > 0)
    if len(xs) < 16: return Image.fromarray(rgb).resize((IMG, IMG))
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    px, py = int(pad * (x1 - x0 + 1)), int(pad * (y1 - y0 + 1)); H, W = rgb.shape[:2]
    x0 = max(0, x0 - px); x1 = min(W - 1, x1 + px); y0 = max(0, y0 - py); y1 = min(H - 1, y1 + py)
    return Image.fromarray(rgb[y0:y1 + 1, x0:x1 + 1]).resize((IMG, IMG))
def mask_B(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, np.uint8)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big).astype(np.uint8)
def crop_B(rgb, m, pad=CROP_PAD):
    ys, xs = np.where(m > 0); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((IMG, IMG), Image.BILINEAR)
RECIPES = {'A_notebook11': (mask_A, crop_A), 'B_repro_fallback': (mask_B, crop_B)}
def jpeg(im):
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=95); buf.seek(0); return Image.open(buf).convert('RGB')

def perturb(m, kind, rng):                              # verbatim logic of notebook 11
    if kind == 'clean': return m
    if kind == 'erode':  return cv2.erode(m, np.ones((ERODE_K, ERODE_K), np.uint8), iterations=1)
    if kind == 'dilate': return cv2.dilate(m, np.ones((DILATE_K, DILATE_K), np.uint8), iterations=1)
    if kind == 'shift':
        h, w = m.shape; M = np.float32([[1, 0, int(round(SHIFT_FRAC * w))], [0, 1, int(round(SHIFT_FRAC * h))]])
        return cv2.warpAffine(m, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
    if kind == 'jitter':
        out = m.copy()
        for op in rng.permutation(['erode', 'dilate', 'shift']):
            if rng.random() < 0.5: out = perturb(out, str(op), rng)
        return out
    raise ValueError(kind)

TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(device).eval()
@torch.no_grad()
def feats(ims, model=None, tf=TO_T, bs=128):
    model = model or dino; out = []
    for i in range(0, len(ims), bs):
        x = torch.stack([tf(im) for im in ims[i:i + bs]]).to(device)
        out.append(torch.nn.functional.normalize(model(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out)

# pick the recipe that reproduces the cached test features
chk = np.random.default_rng(0).choice(len(TEST), 128, replace=False)
COS = {}
for name, (mf, cf) in RECIPES.items():
    ims = []
    for i in chk:
        rgb = np.array(Image.open(TEST[i]['path']).convert('RGB')); ims.append(jpeg(cf(rgb, mf(rgb))))
    c = (feats(ims) * Xte_c[chk]).sum(1); COS[name] = dict(median=float(np.median(c)), min=float(c.min()))
    print('%-18s cosine to cached features: median %.5f, min %.5f' % (name, np.median(c), c.min()))
RECIPE = max(COS, key=lambda k: COS[k]['median']); MASK, CROP = RECIPES[RECIPE]
print('-> using', RECIPE)
assert COS[RECIPE]['median'] > 0.99, 'no recipe reproduces the cache: stop and report'


In [ ]:
# CELL 5 — TIMING (Reviewer 2, Comment 5): crops and DINO features for all 13,084 images
os.makedirs(CACHE, exist_ok=True)
cp = lambda r: os.path.join(CACHE, hashlib.md5(r['rel'].encode()).hexdigest() + '.jpg')
def _build(r):
    if not os.path.exists(cp(r)):
        rgb = np.array(Image.open(r['path']).convert('RGB')); CROP(rgb, MASK(rgb)).save(cp(r), quality=95)
    return True
t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex: list(ex.map(_build, RECS, chunksize=32))
T_CROP = time.time() - t0
print('segment + crop + JPEG, %d images: %.1f s (%d CPU workers)' % (len(RECS), T_CROP, os.cpu_count()))
if device.type == 'cuda': torch.cuda.synchronize()
t0 = time.time()
Xall = feats([Image.open(cp(r)).convert('RGB') for r in RECS])
if device.type == 'cuda': torch.cuda.synchronize()
T_FEAT = time.time() - t0
gpu = torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'
print('DINO ViT-S/16 features, %d images (incl. reading the crops): %.1f s on %s' % (len(RECS), T_FEAT, gpu))
idx = {r['rel']: k for k, r in enumerate(RECS)}
Xte_r = Xall[[idx[r['rel']] for r in TEST]]; Xtr_r = Xall[[idx[r['rel']] for r in TRAIN]]
print('rebuilt vs cached: test median cosine %.5f | train median cosine %.5f'
      % (np.median((Xte_r * Xte_c).sum(1)), np.median((Xtr_r * Xtr_c).sum(1))))


In [ ]:
# CELL 6 — probes (the published five 70-label draws) and a helper
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def probe_on(Xa, Xb):
    auc, acc = [], []
    for rep in range(5):
        i = draw(rep); m = LogisticRegression(C=1.0, max_iter=2000).fit(Xa[i], ytr[i]); P = m.predict_proba(Xb)
        auc.append(roc_auc_score(dte, 1 - P[:, HEALTHY])); acc.append(accuracy_score(yte, P.argmax(1)))
    return dict(auc=[float(np.mean(auc)), float(np.std(auc, ddof=1))], acc7=[float(np.mean(acc)), float(np.std(acc, ddof=1))],
                per_rep_auc=[float(a) for a in auc])
REF = probe_on(Xtr_c, Xte_c); print('cached features (Tables 9, 11): AUC %.4f, 7-class %.3f' % (REF['auc'][0], REF['acc7'][0]))
CLEAN_R = probe_on(Xtr_c, Xte_r); print('cached train / rebuilt test crops: AUC %.4f' % CLEAN_R['auc'][0])


In [ ]:
# CELL 7 — TABLE 2: three encoders on the SAME cached crops (DINOv1 = the cached features themselves)
import timm
T2 = {'DINOv1 ViT-S/16': REF}
SPECS = {'DINOv2 ViT-S/14': ('vit_small_patch14_dinov2.lvd142m', dict(img_size=224), [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
         'CLIP ViT-B/16':   ('vit_base_patch16_clip_224.openai', {}, [0.4815, 0.4578, 0.4082], [0.2686, 0.2613, 0.2758])}
for name, (arch, kw, mu, sd) in SPECS.items():
    m = timm.create_model(arch, pretrained=True, num_classes=0, **kw).eval().to(device)
    tf = T.Compose([T.ToTensor(), T.Normalize(mu, sd)])
    Xa = feats([Image.open(cp(r)).convert('RGB') for r in TRAIN], m, tf)
    Xb = feats([Image.open(cp(r)).convert('RGB') for r in TEST], m, tf)
    T2[name] = probe_on(Xa, Xb); del m; torch.cuda.empty_cache()
for k, v in T2.items(): print('%-16s AUC %.3f ± %.3f | 7-class %.3f' % (k, *v['auc'], v['acc7'][0]))


In [ ]:
# CELL 8 — TABLE 14: mask perturbations on the test leaves, same recipe, probes fixed
def pert_crop(args):
    i, kind = args
    rgb = np.array(Image.open(TEST[i]['path']).convert('RGB'))
    m = perturb(MASK(rgb), kind, np.random.default_rng(i))
    return jpeg(CROP(rgb, m))
T14 = {}
for kind in ('clean', 'erode', 'dilate', 'shift', 'jitter'):
    t0 = time.time()
    if kind == 'clean': Xk = Xte_r
    else:
        with ThreadPoolExecutor(os.cpu_count()) as ex: ims = list(ex.map(pert_crop, [(i, kind) for i in range(len(TEST))]))
        Xk = feats(ims)
    T14[kind] = probe_on(Xtr_c, Xk)
    print('%-7s AUC %.3f ± %.3f  (%.0fs)' % (kind, *T14[kind]['auc'], time.time() - t0))


In [ ]:
# CELL 9 — TABLE 8: synthetic clutter, same recipe, probes fixed (textures verbatim from the original script)
def field_textures(n=40, size=224, seed=0):
    rng = np.random.RandomState(seed); out = []
    pal_ = [[(34, 85, 34), (60, 110, 45), (20, 60, 25)], [(85, 65, 34), (110, 90, 55), (60, 45, 25)],
            [(45, 90, 45), (70, 120, 60), (30, 70, 35)], [(70, 60, 40), (95, 85, 60), (50, 40, 30)]]
    for i in range(n):
        pal = pal_[i % 4]; img = np.zeros((size, size, 3), np.uint8); img[:] = pal[0]
        for _ in range(70):
            x, y = rng.randint(0, size - 20, 2); w, h = rng.randint(15, 70, 2)
            img[y:y + h, x:x + w] = np.clip(np.array(pal[rng.randint(len(pal))]) + rng.randint(-30, 30, 3), 0, 255)
        out.append(cv2.GaussianBlur(img, (9, 9), 3).astype(np.float32))
    return np.stack(out)
BGS = field_textures(); BG = np.random.default_rng(0).integers(0, len(BGS), len(TEST))
def clutter_crop(args):
    i, a = args
    rgb = np.array(Image.open(TEST[i]['path']).convert('RGB'))
    bg = cv2.resize(BGS[BG[i]], (rgb.shape[1], rgb.shape[0]))
    rgb = ((1 - a) * rgb.astype(np.float32) + a * bg).clip(0, 255).astype(np.uint8)
    return jpeg(CROP(rgb, MASK(rgb)))
T8 = {'0.0': CLEAN_R}
for a in ALPHAS[1:]:
    with ThreadPoolExecutor(os.cpu_count()) as ex: ims = list(ex.map(clutter_crop, [(i, a) for i in range(len(TEST))]))
    T8[str(a)] = probe_on(Xtr_c, feats(ims)); print('alpha %.1f AUC %.3f ± %.3f' % (a, *T8[str(a)]['auc']))


In [ ]:
# CELL 10 — save
json.dump(dict(recipe=RECIPE, recipe_cosine=COS, reference_cached=REF, clean_rebuilt=CLEAN_R,
               table2=T2, table14=T14, table8=T8,
               timing=dict(n_images=len(RECS), crop_seconds=T_CROP, cpu_workers=os.cpu_count(),
                           feature_seconds=T_FEAT, device=gpu),
               files=FILES), open(OUT, 'w'), indent=1, default=float)
print('saved', OUT, '-> upload this file')
